# 07 · Orchestration

Pablo's LangGraph wiring with the real Merchant and Humanizer nodes in place of his stubs:

```
INTENT → CLARIFY? → SEARCHING → EVALUATING → RANKING → RESPONDING → DONE
                       ↑______________| (once)            ↺ (once)
```

All agents share one LLM client and one listing pool. Every node can still be replaced,
which is how the tests run it with fakes. `ChatSession` runs a conversation: it pauses at
CLARIFY, sends the answer back in, and introduces the assistant as an AI on the first turn.

Run notebooks 00–06 once (or just their export cells) after changing them; this one only
imports the modules.

In [ ]:
#| default_exp graph

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib.util
import os
import sys
from pathlib import Path

# Stop early with a clear message if VS Code picked the wrong kernel.
missing = [
    m for m in ("yaml", "dotenv", "langgraph", "pydantic") if not importlib.util.find_spec(m)
]
if missing:
    raise ModuleNotFoundError(
        f"This kernel ({sys.executable}) is not the project environment; missing {missing}. "
        "Run `uv sync` in the repo, then click the kernel name at the top right of the "
        "notebook -> Select Another Kernel -> Python Environments -> .venv (Python 3.14)."
    )

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets this notebook import the src package

LIVE = False  # True: real Gemini calls with the keys in .env. False: offline, free, no keys.
os.environ["AGENTIC_LLM"] = "primary" if LIVE else "offline"

In [ ]:
#| export
"""LangGraph 7-state orchestration graph (Section 3 & 8).

States: INTENT -> CLARIFY (optional) -> SEARCHING -> EVALUATING -> RANKING -> RESPONDING -> DONE
Bounded loops:
  1. INTENT -> CLARIFY (at most 1 question) -> INTENT (once user answers) or pause
  2. EVALUATING -> SEARCHING (at most 1 retry with relaxed filters if 0 listings survive)
  3. RESPONDING -> RESPONDING (at most 1 retry if number check fails, then template fallback)
"""

from collections.abc import Callable
from pathlib import Path
from typing import Any

from langgraph.graph import END, START, StateGraph

from src.agents.consumer import make_done_node, make_intent_node, make_ranking_node
from src.agents.humanizer import make_responding_node
from src.agents.merchant import distinct_printings, make_evaluating_node, make_searching_node
from src.agents.platform import (
    make_clarify_node,
    route_after_evaluating,
    route_after_intent,
    route_after_responding,
)
from src.config import load_config
from src.llm import LLM, make_llm
from src.memory import BuyerMemory
from src.pipeline import MarketData, load_data
from src.state import AgentState

In [ ]:
#| export
Node = Callable[[AgentState], dict[str, Any]]

In [ ]:
#| export
def build_graph(
    config: dict[str, Any] | None = None,
    memory: BuyerMemory | None = None,
    llm: LLM | None = None,
    data: MarketData | None = None,
    intent_node: Node | None = None,
    clarify_node: Node | None = None,
    searching_node: Node | None = None,
    evaluating_node: Node | None = None,
    ranking_node: Node | None = None,
    responding_node: Node | None = None,
    done_node: Node | None = None,
    pause_on_clarify: bool = True,
):
    """Compile and return the LangGraph StateGraph.

    Every node can be replaced (tests pass fakes). By default all agents share one LLM
    (config `llm.active`, or env AGENTIC_LLM, e.g. AGENTIC_LLM=offline) and one listing pool.

    Set pause_on_clarify=True for interactive CLI / scripted clarify turns (CLARIFY -> END so the
    caller can collect clarify_answer and re-enter INTENT). ChatSession does this for you.
    """
    cfg = config or load_config()
    mem = memory or BuyerMemory()
    model = llm or make_llm(cfg)
    pool = data or load_data(cfg)

    max_clarify = int(cfg["limits"]["max_clarify_questions"])
    max_search_retries = int(cfg["limits"]["max_search_retries"])
    max_humanizer_retries = int(cfg["limits"]["max_humanizer_retries"])
    risk_reject = float(cfg["thresholds"]["risk_reject"])

    def printings(q) -> list[str]:
        return distinct_printings(pool.listings, q.card_name, cfg)

    graph = StateGraph(AgentState)

    graph.add_node("INTENT", intent_node or make_intent_node(memory=mem, config=cfg, llm=model))
    graph.add_node(
        "CLARIFY", clarify_node or make_clarify_node(config=cfg, llm=model, printings_fn=printings)
    )
    graph.add_node("SEARCHING", searching_node or make_searching_node(pool, cfg, llm=model))
    graph.add_node("EVALUATING", evaluating_node or make_evaluating_node(pool, cfg, llm=model))
    graph.add_node("RANKING", ranking_node or make_ranking_node(config=cfg))
    graph.add_node(
        "RESPONDING",
        responding_node
        or make_responding_node(cfg, llm=model, sample_data=pool.source == "samples"),
    )
    graph.add_node("DONE", done_node or make_done_node(memory=mem))

    graph.add_edge(START, "INTENT")

    graph.add_conditional_edges(
        "INTENT",
        lambda s: route_after_intent(s, max_clarify=max_clarify, printings_fn=printings),
        {"CLARIFY": "CLARIFY", "SEARCHING": "SEARCHING"},
    )

    if pause_on_clarify:
        graph.add_edge("CLARIFY", END)
    else:
        graph.add_edge("CLARIFY", "SEARCHING")

    graph.add_edge("SEARCHING", "EVALUATING")

    graph.add_conditional_edges(
        "EVALUATING",
        lambda s: route_after_evaluating(
            s,
            risk_reject_threshold=risk_reject,
            max_search_retries=max_search_retries,
        ),
        {"SEARCHING": "SEARCHING", "RANKING": "RANKING"},
    )

    graph.add_edge("RANKING", "RESPONDING")

    graph.add_conditional_edges(
        "RESPONDING",
        lambda s: route_after_responding(s, max_humanizer_retries=max_humanizer_retries),
        {"RESPONDING": "RESPONDING", "DONE": "DONE"},
    )

    graph.add_edge("DONE", END)

    return graph.compile()

## Conversations

In [ ]:
#| export
class ChatSession:
    """One conversation with the graph: handles the clarify pause and says it is an AI
    assistant at the start of the session (Section 6).

        chat = ChatSession(build_graph())
        reply, state = chat.send("Base Set Charizard under $400")
    """

    def __init__(self, app, session_id: str = "default", config: dict[str, Any] | None = None):
        self.app = app
        self.session_id = session_id
        self.cfg = config or load_config()
        self.pending: str | None = None  # request waiting for the answer to our question
        self.turns = 0

    def send(self, text: str) -> tuple[str, AgentState]:
        base: AgentState = {
            "session_id": self.session_id,
            "search_retry_count": 0,
            "humanizer_retry_count": 0,
        }
        if self.pending is not None:
            state: AgentState = {
                **base, "user_input": self.pending, "clarify_answer": text, "clarify_count": 1
            }  # fmt: skip
        else:
            state = {**base, "user_input": text, "clarify_count": 0}
        result = self.app.invoke(state)
        if result.get("current_state") == "CLARIFY":
            self.pending = state["user_input"]
            reply = result["clarifying_question"]
        else:
            self.pending = None
            reply = result["final_response"]
        if self.turns == 0:
            reply = f"{self.cfg['response']['ai_disclosure']} {reply}"
        self.turns += 1
        return reply, result

In [ ]:
#| export
def export_architecture_diagram(output_path: Path | str = "docs/architecture.png") -> Path:
    """Export the LangGraph Mermaid architecture figure for the report (Section 8)."""
    compiled = build_graph(llm=make_llm(load_config(), "offline"))
    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    png_bytes = compiled.get_graph().draw_mermaid_png()
    out.write_bytes(png_bytes)
    return out

# Try it

The graph, as Mermaid text (paste into mermaid.live, or use `export_architecture_diagram()` online):

In [ ]:
import json
import tempfile

import pandas as pd

from src.config import load_config, repo_path
from src.llm import make_llm
from src.memory import BuyerMemory

cfg = load_config()
llm = make_llm(cfg)
app = build_graph(cfg, llm=llm, memory=BuyerMemory(Path(tempfile.mkdtemp()) / "demo.sqlite"))
print(app.get_graph().draw_mermaid())

Every sample query end to end (Q03 gets the clarifying question and its scripted answer):

In [ ]:
queries = json.loads(repo_path("data/samples/sample_queries.json").read_text(encoding="utf-8"))
rows = []
for q in queries:
    chat = ChatSession(app, q["id"], cfg)
    reply, state = chat.send(q["request"])
    if state["current_state"] == "CLARIFY":
        reply, state = chat.send(q["clarify_answer"])
    ranked = state.get("ranked_listings") or []
    top = ranked[0].listing.listing_id if ranked else None
    ok = {
        "recommend": top in q["acceptable"],
        "nothing_fits": top is None,
        "price_check": "usually goes for" in reply,
    }[q["expect_status"]]
    rows.append(
        {
            "id": q["id"],
            "expect": q["expect_status"],
            "acceptable": q["acceptable"],
            "top": top,
            "ok": ok,
            "source": state.get("response_source"),
            "reply": reply,
        }
    )
pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

Where the time goes in one request:

In [ ]:
_, state = ChatSession(app, "timing", cfg).send(
    "Base Set Charizard under $400, I care about condition"
)
print(state["stage_timings"])
print(llm.usage())

Chat in a loop (type `quit` to stop). Set `INTERACTIVE = True` to try it.

In [ ]:
INTERACTIVE = False
if INTERACTIVE:
    chat = ChatSession(app, "me", cfg)
    while (text := input("you: ").strip()) and text.lower() != "quit":
        reply, _ = chat.send(text)
        print("assistant:", reply)

## Export

Run this after you change any `#| export` cell. It rewrites the generated module in `src/`,
so the other notebooks, the graph and the tests use your new version.

In [ ]:
from src.nbexport import export_all

changed = export_all()